# Composition-descriptor models for oxide glasses
### Young's modulus · density · liquidus temperature: XGBoost and symbolic regression

**How to run:** *Restart Kernel → Run All*. All cells depend on the ones above them.

1. **First time: leave `SELF_TEST = True`.** The notebook runs on a small subsample and checks every part with your real libraries:
   - descriptors
   - folds
   - XGBoost and Optuna
   - PySR
   - saving and re-loading, including an interrupt-and-resume test

   It ends with a PASS/FAIL table. It takes about 5–15 min; the first PySR import also installs Julia once.
2. **Then set `SELF_TEST = False` and Run All** for the full run.

**Stopping and resuming.** Everything is saved to disk as soon as it finishes:
- every Optuna trial
- every outer fold
- every PySR run (one per setting and fold)
- every final model

If the run stops (crash, restart, power cut), *Restart Kernel → Run All* again. Finished trials, folds and PySR runs are loaded from disk and **not recomputed**. At most the one trial or PySR run that was in progress is repeated. Progress is printed with timestamps and also written to `progress.log`.

**Data are fetched automatically.**
- The datasets are downloaded from the authors' public repository, [kzzzynobad/High-Specific-Modulus-Glass](https://github.com/kzzzynobad/High-Specific-Modulus-Glass), and verified by checksum.
- The literature pure-oxide table is embedded in the notebook and written to `pure_oxide_properties.csv` on the first run. To fill a gap, edit that CSV, adding a value **and its source**.

**Descriptors.** Each property $P$ gets five descriptors built from the mole fractions $x_i$ and the pure-oxide values $p_i$ of that same property:

| Descriptor | Formula |
|---|---|
| `mean_p` | $\sum_i x_i p_i$ |
| `gmean_p` | $\exp(\sum_i x_i \ln p_i)$ |
| `hmean_p` | $1/\sum_i (x_i/p_i)$ |
| `var_p` | $\sum_i x_i p_i^2 - (\sum_i x_i p_i)^2$ |
| `S_mix` | $-\sum_i x_i \ln x_i$ |

**Evaluation.** Nested, grouped k-fold cross-validation:
- the outer folds give the reported scores;
- the inner folds tune XGBoost (Optuna) and choose the PySR settings and equation complexity;
- glasses with identical descriptor vectors always stay in the same fold.

Baselines on the same folds: the mixing rule alone, linear regression, and XGBoost on the raw oxide fractions.

## 0. Check installed packages

In [ ]:
# If something is missing, install it into THIS kernel's environment, then restart the kernel:
# %pip install "numpy>=1.24" "pandas>=2.0" "scikit-learn>=1.3" "xgboost>=2.0" "optuna>=3.4" "pysr>=1.0" sympy matplotlib
import importlib.util, sys
_required = ["numpy", "pandas", "sklearn", "matplotlib", "xgboost", "optuna", "sympy", "pysr"]
_missing = [m for m in _required if importlib.util.find_spec(m) is None]
print("Python:", sys.executable)
if _missing:
    raise ImportError(f"Missing packages in this kernel: {_missing}. Run the %pip line above (uncommented), "
                      "then Restart Kernel and Run All.")
print("All required packages found. (shap is optional.)")

## 1. Configuration

In [ ]:
from pathlib import Path

SELF_TEST = True        # True: quick end-to-end verification on a subsample (do this first). False: full run.

# ---------------- data ----------------
GITHUB_RAW     = "https://raw.githubusercontent.com/kzzzynobad/High-Specific-Modulus-Glass/main/DATASET"
LOCAL_DATA_DIR = Path("DATASET")        # used instead of downloading if this folder exists
CACHE_DIR      = Path("data_cache")
OXIDE_TABLE    = Path("pure_oxide_properties.csv")
RESULTS_DIR    = Path("results_v2")
RUN_PROPERTIES = None                   # None = all; or e.g. ["density"] to run one property at a time

# ---------------- cross-validation / XGBoost ----------------
SEED          = 42
OUTER_FOLDS   = 5
INNER_FOLDS   = 5
XGB_TRIALS    = 100                     # Optuna trials per outer fold (and for the final model)
XGB_MAX_DEPTH = 8
N_JOBS        = -1                      # CPU threads for XGBoost
MIN_FREE_RAM_GB = 1.0                   # below this much free RAM the run stops cleanly (progress is saved) instead of failing trials
RUN_XGB       = False                   # XGBoost stage (nested CV + final model). False = only baselines + symbolic regression
RUN_RAW_OXIDE_BASELINE = True           # XGBoost on the 17 raw mole fractions (same rows, same folds)

# ---------------- symbolic regression (PySR) ----------------
RUN_SR             = True
SR_NITERATIONS     = 1000
SR_TIMEOUT_MIN     = 20                 # wall-clock cap per PySR run (minutes)
SR_POPULATIONS     = 31
SR_POPULATION_SIZE = 50
SR_JULIA_HEAP_GB   = 4                  # soft memory limit for PySR's Julia process
SR_VAL_FRACTION    = 0.2                # grouped inner split used to select settings + equation complexity
SR_TOLERANCE       = 0.01               # simplest equation within 1 % of the best validation RMSE
SR_GRID = [dict(maxsize=25, parsimony=1e-4),
           dict(maxsize=30, parsimony=1e-5),
           dict(maxsize=35, parsimony=0.0)]
SR_BINARY_OPS = ["+", "-", "*", "/", "^"]
SR_UNARY_OPS  = ["square", "cube", "sqrt", "log", "exp"]
SR_BATCH_SIZE = 2000                    # mini-batching is used automatically above 5000 training rows
SR_DETERMINISTIC = False                # True = bit-reproducible but single-threaded (much slower)

# ---------------- final models ----------------
RUN_FINAL = True

# ---------------- descriptor construction ----------------
MISSING_POLICY = "drop"                 # "drop": remove glasses containing an oxide without a value; "renormalize": sensitivity check only
PROPERTY_OVERRIDES = {}                 # e.g. {"melting_point_K": {"K2O": 623.15}} -- report any override + source in the paper

if SELF_TEST:
    RESULTS_DIR = Path("results_selftest")
    SELFTEST_ROWS = 500                 # rows per property
    OUTER_FOLDS, INNER_FOLDS, XGB_TRIALS = 3, 3, 6
    SR_NITERATIONS, SR_TIMEOUT_MIN = 20, 1
    SR_GRID = SR_GRID[:2]
    SR_POPULATIONS, SR_POPULATION_SIZE = 8, 20

TARGETS = {   # dataset folder, target column, oxide-table column, unit, MD5 of the published CLEAN.csv
    "youngs_modulus":       dict(folder="YOUNGSMODULUS", target="YoungModulus", prop="youngs_modulus_GPa", unit="GPa",
                                 md5="e9843ed350120cd31fc482d6f2335f65"),
    "density":              dict(folder="DENSITY",       target="Density293K",  prop="density_g_cm3",      unit="g/cm³",
                                 md5="b7da382a7058ea7b80441449bb0d6326"),
    "liquidus_temperature": dict(folder="TLIQUIDUS",     target="Tliquidus",    prop="melting_point_K",    unit="K",
                                 md5="adece4e478c7181525861e514b37dedd"),
}
if RUN_PROPERTIES is not None:
    TARGETS = {k: TARGETS[k] for k in RUN_PROPERTIES}
FEATURES = ["mean_p", "gmean_p", "hmean_p", "var_p", "S_mix"]
CONFIG_LOADED = True

## 2. Imports, logging, safe file writing

In [ ]:
assert "CONFIG_LOADED" in globals(), "Run the notebook from the top: Restart Kernel -> Run All"
import json, os, platform, random, time, warnings, hashlib, gc, shutil, urllib.request
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, median_absolute_error
import xgboost as xgb
import optuna
import sympy as sp

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*random_state.*deterministic.*")
warnings.filterwarnings("ignore", message=".*large maxsize.*")
optuna.logging.set_verbosity(optuna.logging.WARNING)
random.seed(SEED); np.random.seed(SEED)

if SELF_TEST and RESULTS_DIR.exists():
    shutil.rmtree(RESULTS_DIR)          # the self-test always starts from scratch
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR = RESULTS_DIR / "checkpoints"
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 10})
XGB_ERRORS = (getattr(xgb.core, "XGBoostError", RuntimeError), MemoryError)

class OutOfMemoryStop(Exception):
    pass

def ram_status():
    # (free system RAM in GB, RAM used by this Python process in GB); (None, None) if psutil is missing
    try:
        import psutil
        return psutil.virtual_memory().available / 1024**3, psutil.Process().memory_info().rss / 1024**3
    except Exception:
        return None, None

def ensure_memory(where):
    free, _ = ram_status()
    if free is not None and free < MIN_FREE_RAM_GB:
        gc.collect(); time.sleep(5)
        free, proc = ram_status()
        if free < MIN_FREE_RAM_GB:
            raise OutOfMemoryStop(f"only {free:.2f} GB RAM free at {where} (this notebook uses {proc:.2f} GB)")

def stop_for_memory(label, err):
    msg = (f"STOPPED: the computer ran out of memory during {label} ({err}). All finished trials and folds are saved. "
           "Close other notebooks/programs (each open Jupyter kernel keeps its RAM), then Restart Kernel and Run All: "
           "the run continues where it stopped. If it happens again, set N_JOBS = 4 in the configuration cell.")
    log(msg)
    raise RuntimeError(msg) from None
T_START = time.time()

def log(*args):
    msg = f"[{datetime.now():%Y-%m-%d %H:%M:%S} | {(time.time() - T_START) / 60:6.1f} min] " + " ".join(map(str, args))
    print(msg, flush=True)
    with open(RESULTS_DIR / "progress.log", "a", encoding="utf-8") as f:
        f.write(msg + "\n")

def atomic_write_text(path, text):
    # Write to a temp file, then rename: an interruption never leaves a half-written file.
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(text, encoding="utf-8")
    for attempt in range(10):            # Windows antivirus / indexers can lock a fresh file briefly
        try:
            os.replace(tmp, path); return
        except PermissionError:
            if attempt == 9: raise
            time.sleep(0.2 * (attempt + 1))

def save_json(path, obj):
    atomic_write_text(path, json.dumps(obj, default=lambda o: o.item() if hasattr(o, "item") else str(o)))

def load_json(path):
    path = Path(path)
    if not path.exists():
        return None
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except json.JSONDecodeError:
        log("WARNING: unreadable checkpoint, recomputing:", path)
        return None

def fingerprint(*items):
    return hashlib.md5(json.dumps(items, sort_keys=True, default=str).encode()).hexdigest()[:10]

def _ver(m):
    try: return __import__(m).__version__
    except Exception: return "not installed"
ENV = {"python": platform.python_version(), "platform": platform.platform(), "self_test": SELF_TEST, "seed": SEED,
       **{m: _ver(m) for m in ["numpy", "pandas", "sklearn", "xgboost", "optuna", "pysr", "sympy", "matplotlib"]}}
save_json(RESULTS_DIR / "environment.json", ENV)
log("results folder:", RESULTS_DIR.resolve(), "| SELF_TEST =", SELF_TEST)
ENV

## 3. Data (downloaded and verified automatically)

In [ ]:
assert "CONFIG_LOADED" in globals(), "Run the notebook from the top: Restart Kernel -> Run All"

def md5sum(path):
    return hashlib.md5(Path(path).read_bytes()).hexdigest()

def get_dataset(cfg):
    local = LOCAL_DATA_DIR / cfg["folder"] / "CLEAN.csv"
    if local.exists():
        path, origin = local, "local folder"
    else:
        path, origin = CACHE_DIR / cfg["folder"] / "CLEAN.csv", "cache"
        if not path.exists() or md5sum(path) != cfg["md5"]:
            path.parent.mkdir(parents=True, exist_ok=True)
            url = f"{GITHUB_RAW}/{cfg['folder']}/CLEAN.csv"
            log("downloading", url)
            urllib.request.urlretrieve(url, path)
            origin = "downloaded"
    if md5sum(path) != cfg["md5"]:
        raise ValueError(f"{path}: checksum differs from the published dataset; results would not be comparable.")
    return path, origin

DATA_FILES = {}
for name, cfg in TARGETS.items():
    DATA_FILES[name], origin = get_dataset(cfg)
    log(f"{name:22s} {DATA_FILES[name]} ({origin}, checksum OK)")

In [ ]:
OXIDE_TABLE_CSV = r"""oxide,density_g_cm3,density_source,melting_point_C,melting_point_K,melting_point_source,youngs_modulus_GPa,youngs_modulus_form,youngs_modulus_source,notes
Li2O,2.013,CRC 85th ed. p.4-66,1570.0,1843.15,CRC 85th ed. p.4-66,159.5,"single crystal (DFT, Hill)","de Jong et al. 2015 Sci. Data 2:150009 (Materials Project; dataset elastic_tensor_2015): K_VRH=78.875 GPa, Poisson ratio=0.163; E=3K(1-2nu)",Newer CRC editions list mp 1438 C; DFT (GGA) value; no measured value available. GGA underestimates E of oxides by ~7-10% (e.g. CaO 180 vs 194 GPa measured)
Na2O,2.27,CRC 85th ed. p.4-85,1132.0,1405.15,CRC 85th ed. p.4-85 (dec),74.7,"single crystal (DFT, Hill)","de Jong et al. 2015 Sci. Data 2:150009 (Materials Project; dataset elastic_tensor_2015): K_VRH=45.944 GPa, Poisson ratio=0.229; E=3K(1-2nu)",Decomposes; not a true melting point; DFT (GGA) value; no measured value available. GGA underestimates E of oxides by ~7-10% (e.g. CaO 180 vs 194 GPa measured)
K2O,2.35,CRC 85th ed. p.4-77,350.0,623.15,"CRC 85th ed. p.4-77 (350 dec): decomposition temperature, K2O has no true melting point",32.1,"single crystal (DFT, Hill)","de Jong et al. 2015 Sci. Data 2:150009 (Materials Project; dataset elastic_tensor_2015): K_VRH=27.264 GPa, Poisson ratio=0.304; E=3K(1-2nu)",Decomposes at 350 C; no melting point exists; DFT (GGA) value; no measured value available. GGA underestimates E of oxides by ~7-10% (e.g. CaO 180 vs 194 GPa measured)
MgO,3.6,CRC 85th ed. p.4-67,2826.0,3099.15,IUPAC 1982 recommended (2826 +/- 30),309.7,polycrystal,NISTIR 6853 (Soga & Anderson 1966; 0.06% porosity),CRC 85th mp 2825
CaO,3.34,CRC 85th ed. p.4-49,2617.0,2890.15,Panek measurement reported in IUPAC 1982 (2617 +/- 30; IUPAC gave no recommended value),193.9,single crystal (Hill),Speziale et al. 2006 JGR 111: Ks=112.0 G=80.05 GPa -> E=9KG/(3K+G),CRC 85th lists mp 2898
SrO,5.1,CRC 85th ed. p.4-87,2531.0,2804.15,CRC 85th ed. p.4-87,151.8,single crystal (Hill),Computed from Cij of Johnson Thrasher & Kearney 1970 J.Appl.Phys.41:427 (C11=177.6 C12=41.3 C44=59.5 GPa) as quoted by UCL crystal database,Secondary source for Cij - verify if paper becomes available
BaO,5.72,CRC 85th ed. p.4-44,1972.0,2245.15,CRC 85th ed. p.4-44,95.6,"single crystal (DFT, Hill)","de Jong et al. 2015 Sci. Data 2:150009 (Materials Project; dataset elastic_tensor_2015): K_VRH=68.357 GPa, Poisson ratio=0.2668; E=3K(1-2nu)",DFT (GGA) value; no measured value available. GGA underestimates E of oxides by ~7-10% (e.g. CaO 180 vs 194 GPa measured)
ZnO,5.6,CRC 85th ed. p.4-95,1974.0,2247.15,CRC 85th ed. p.4-95,119.0,polycrystal,NISTIR 6853 (Martin Dadon & Rosen 1996; 0.5% porosity),
Al2O3,3.97,CRC 85th ed. p.4-39,2054.0,2327.15,IUPAC 1982 recommended (2054 +/- 6),396.0,polycrystal,NISTIR 6853 (Lang 1960; rho=3.942),CRC 85th mp 2053
SiO2,2.196,CRC 85th ed. p.4-82 (vitreous),1726.0,1999.15,IUPAC 1982 recommended (1726 +/- 10),73.0,glass,NISTIR 6853 (Spinner 1954; McSkimin 1959; 72.9-73.1),Rouxel 2007 lists 70 GPa for Vitreosil; alpha-quartz density 2.648
B2O3,2.55,CRC 85th ed. p.4-47,450.0,723.15,CRC 85th ed. p.4-47,17.4,glass,Rouxel 2007 J.Am.Ceram.Soc. 90:3019 Table I,CRC density is for hex crystal
P2O5,2.3,CRC 85th ed. p.4-74,562.0,835.15,CRC 85th ed. p.4-74,31.3,glass,Rouxel 2007 J.Am.Ceram.Soc. 90:3019 Table I,CRC density is for orthorhombic crystal
TiO2,4.23,CRC 85th ed. p.4-91,1843.0,2116.15,IUPAC 1982 recommended (1843 +/- 15 in air; erratum value),272.0,polycrystal (rutile),NISTIR 6853 (Minnear & Bradt 1977; 0.02% porosity),
Fe2O3,5.25,CRC 85th ed. p.4-63,1565.0,1838.15,CRC 85th ed. p.4-63,234.7,polycrystal (ultrasonic),Zou et al. 2025 iScience 28:111905 (doi:10.1016/j.isci.2025.111905): KS0=235.4 G0=88.0 GPa -> E=9KG/(3K+G),Measured; agrees with Liebermann & Schreiber G0=91.0 GPa
Y2O3,5.03,CRC 85th ed. p.4-95,2439.0,2712.15,IUPAC 1982 recommended (2439 +/- 12),171.5,polycrystal (zero porosity),NISTIR 6853 (Manning Hunter & Powell 1969; extrapolated to 0% porosity),Measured at 0.04% porosity: 156-158 GPa
ZrO2,5.68,CRC 85th ed. p.4-96,2710.0,2983.15,IUPAC 1982 recommended (2710 +/- 35),197.0,polycrystal (monoclinic),NISTIR 6853 (Smith & Crandall 1964; 0.05% porosity),Dataset column is named ZrO but means ZrO2
SnO2,6.85,CRC 85th ed. p.4-91,1630.0,1903.15,CRC 85th ed. p.4-91,263.5,single crystal (Hill),Computed from measured Cij of Chang & Graham 1975 J. Geophys. Res. 80:2595 (C11=261.7 C12=177.2 C13=155.5 C33=449.6 C44=103.7 C66=207.4 GPa; as tabulated in arXiv:1108.0245),Voigt-Reuss-Hill: K=212.3 G=101.9 GPa
"""

from pathlib import Path
OXIDE_TABLE = globals().get("OXIDE_TABLE", Path("pure_oxide_properties.csv"))
if not OXIDE_TABLE.exists():
    OXIDE_TABLE.write_text(OXIDE_TABLE_CSV.strip() + "\n", encoding="utf-8")
    print("wrote the embedded literature table to", OXIDE_TABLE.resolve())
else:
    print("using existing", OXIDE_TABLE.resolve(), "(edit it to fill gaps; delete it to restore the embedded version)")

In [ ]:
assert "CONFIG_LOADED" in globals(), "Run the notebook from the top: Restart Kernel -> Run All"
ox = pd.read_csv(OXIDE_TABLE).set_index("oxide")
for prop, vals in PROPERTY_OVERRIDES.items():
    for oxide, v in vals.items():
        log(f"OVERRIDE {prop}[{oxide}] = {v} (table: {ox.loc[oxide, prop]})")
        ox.loc[oxide, prop] = v
COLUMN_RENAME = {"ZrO": "ZrO2"}             # dataset column 'ZrO' is zirconia
_props = [t["prop"] for t in TARGETS.values()]
assert (ox[_props].dropna() > 0).all().all(), "pure-oxide values must be positive"
OXIDE_TABLE_HASH = md5sum(OXIDE_TABLE) + fingerprint(PROPERTY_OVERRIDES)

def load_dataset(name, cfg):
    df = pd.read_csv(DATA_FILES[name]).rename(columns=COLUMN_RENAME)
    y = df[cfg["target"]].astype(float).to_numpy()
    comp = df.drop(columns=[cfg["target"]]).astype(float)
    unknown = set(comp.columns) - set(ox.index)
    assert not unknown, f"oxides missing from the oxide table: {unknown}"
    assert (comp.to_numpy() >= 0).all() and (comp.sum(axis=1) > 0).all()
    return comp.div(comp.sum(axis=1), axis=0), y          # mole fractions summing to 1

raw = {name: load_dataset(name, cfg) for name, cfg in TARGETS.items()}
display(ox[_props])
pd.DataFrame({n: {"rows": len(y), "mean": y.mean(), "std": y.std(), "min": y.min(), "max": y.max()} for n, (c, y) in raw.items()}).T

## 4. Descriptors

In [ ]:
assert "CONFIG_LOADED" in globals(), "Run the notebook from the top: Restart Kernel -> Run All"

def build_descriptors(comp, p, policy="drop"):
    p = p.reindex(comp.columns)
    X = comp.to_numpy()
    missing = p.isna().to_numpy()
    if policy == "drop":
        keep = ~(X[:, missing] > 0).any(axis=1)
        Xk = X[keep][:, ~missing]
    elif policy == "renormalize":
        Xk = X[:, ~missing]; keep = Xk.sum(axis=1) > 0
        Xk = Xk[keep] / Xk[keep].sum(axis=1, keepdims=True)
    else:
        raise ValueError(policy)
    pv = p.to_numpy()[~missing]
    mean = Xk @ pv
    D = pd.DataFrame({
        "mean_p": mean,
        "gmean_p": np.exp(Xk @ np.log(pv)),
        "hmean_p": 1.0 / (Xk @ (1.0 / pv)),
        "var_p": np.clip(Xk @ pv**2 - mean**2, 0, None),
        "S_mix": -(np.where(Xk > 0, Xk * np.log(np.where(Xk > 0, Xk, 1.0)), 0.0)).sum(axis=1),
    })
    return D, keep, {"rows_in": len(comp), "rows_kept": int(keep.sum()), "rows_dropped": int((~keep).sum()),
                     "oxides_without_value": list(p.index[missing])}

# --- built-in correctness checks on synthetic compositions ---
_p = pd.Series({"A": 2.0, "B": 8.0})
_D, _, _ = build_descriptors(pd.DataFrame({"A": [1.0, 0.5], "B": [0.0, 0.5]}), _p)
assert np.allclose(_D.iloc[0], [2, 2, 2, 0, 0])                                   # pure oxide: means = p, var = S = 0
assert np.allclose(_D.iloc[1], [5, 4, 3.2, 9, np.log(2)])                          # 50/50 mix, hand-computed
assert (_D["hmean_p"] <= _D["gmean_p"] + 1e-12).all() and (_D["gmean_p"] <= _D["mean_p"] + 1e-12).all()

data, build_report = {}, {}
for name, cfg in TARGETS.items():
    comp, y = raw[name]
    D, keep, rep = build_descriptors(comp, ox[cfg["prop"]], MISSING_POLICY)
    assert (D["hmean_p"] <= D["gmean_p"] * (1 + 1e-9)).all() and (D["gmean_p"] <= D["mean_p"] * (1 + 1e-9)).all()
    d = dict(X=D.reset_index(drop=True), y=y[keep], comp=comp.loc[keep].reset_index(drop=True))
    if SELF_TEST and len(d["y"]) > SELFTEST_ROWS:
        idx = np.sort(np.random.default_rng(SEED).choice(len(d["y"]), SELFTEST_ROWS, replace=False))
        d = dict(X=d["X"].iloc[idx].reset_index(drop=True), y=d["y"][idx], comp=d["comp"].iloc[idx].reset_index(drop=True))
        rep["self_test_subsample"] = SELFTEST_ROWS
    data[name], build_report[name] = d, rep
    d["X"].assign(target=d["y"]).to_csv(RESULTS_DIR / f"descriptors_{name}.csv", index=False)
log("descriptors built")
pd.DataFrame(build_report).T

Check the table above. If many glasses were dropped, the models only cover the composition space that remains, and the paper has to say so. The proper fix is to fill the missing values in `pure_oxide_properties.csv`.

In [ ]:
assert "CONFIG_LOADED" in globals(), "Run the notebook from the top: Restart Kernel -> Run All"

def descriptor_groups(X, decimals=8):
    return pd.factorize(X.round(decimals).astype(str).agg("|".join, axis=1))[0]

for name, d in data.items():
    d["groups"] = descriptor_groups(d["X"])
    log(f"{name:22s} rows={len(d['y'])}  unique descriptor vectors={d['groups'].max() + 1}")

for name, d in data.items():
    X, y = d["X"], d["y"]
    fig, axes = plt.subplots(1, len(FEATURES) + 1, figsize=(3 * (len(FEATURES) + 1), 2.8))
    for ax, f in zip(axes, FEATURES):
        ax.scatter(X[f], y, s=2, alpha=0.3); ax.set_xlabel(f); ax.set_ylabel(TARGETS[name]["target"])
    c = pd.concat([X, pd.Series(y, name="target")], axis=1).corr(method="spearman")
    axes[-1].imshow(c, vmin=-1, vmax=1, cmap="RdBu_r"); axes[-1].set_title("Spearman ρ")
    axes[-1].set_xticks(range(len(c))); axes[-1].set_xticklabels(c.columns, rotation=90, fontsize=7)
    axes[-1].set_yticks(range(len(c))); axes[-1].set_yticklabels(c.columns, fontsize=7)
    fig.suptitle(name); fig.tight_layout(); fig.savefig(RESULTS_DIR / f"descriptors_{name}.png"); plt.show(); plt.close(fig)

## 5. Cross-validation utilities

In [ ]:
assert "CONFIG_LOADED" in globals(), "Run the notebook from the top: Restart Kernel -> Run All"

def grouped_kfold(groups, k, seed):
    uniq = np.unique(groups)
    fold_of_group = pd.Series(np.arange(len(uniq)) % k, index=np.random.default_rng(seed).permutation(uniq))
    fold = fold_of_group.reindex(groups).to_numpy()
    return [(np.where(fold != i)[0], np.where(fold == i)[0]) for i in range(k)]

def grouped_split(groups, val_fraction, seed):
    uniq = np.unique(groups)
    val_groups = np.random.default_rng(seed).choice(uniq, size=max(1, int(round(val_fraction * len(uniq)))), replace=False)
    is_val = np.isin(groups, val_groups)
    return np.where(~is_val)[0], np.where(is_val)[0]

def metrics(y_true, y_pred):
    return {"R2": r2_score(y_true, y_pred), "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
            "MAE": mean_absolute_error(y_true, y_pred), "MedAE": median_absolute_error(y_true, y_pred)}

def summarize(rows):
    return pd.DataFrame(rows).groupby("model")[["R2", "RMSE", "MAE", "MedAE"]].agg(["mean", "std"])

def index_hash(idx):
    return hashlib.md5(np.asarray(idx, dtype=np.int64).tobytes()).hexdigest()[:12]

for name, d in data.items():
    d["outer"] = grouped_kfold(d["groups"], OUTER_FOLDS, SEED)
    for tr, te in d["outer"]:
        assert not set(d["groups"][tr]) & set(d["groups"][te]), "group leak across outer split"
    assert sorted(np.concatenate([te for _, te in d["outer"]]).tolist()) == list(range(len(d["y"])))
log("folds built; no group appears on both sides of any split")

## 6. XGBoost: nested grouped CV, Optuna tuning with trial-level resume

Every finished Optuna trial is appended to a JSON log. On a re-run, finished trials are loaded back into the study and only the remaining trials run. The loaded trials keep their parameters, scores and pruning history, so the TPE sampler still learns from them.

A resumed search draws new random numbers (the sampler seed is offset by the number of finished trials), so it is equally valid but not bit-identical to an uninterrupted run.

In [ ]:
assert "CONFIG_LOADED" in globals(), "Run the notebook from the top: Restart Kernel -> Run All"
from optuna.distributions import IntDistribution, FloatDistribution
from optuna.trial import create_trial, TrialState

XGB_SPACE = {   # name: (type, low, high, log-scale)
    "n_estimators":     ("int",   100, 3000, True),
    "learning_rate":    ("float", 0.005, 0.3, True),
    "max_depth":        ("int",   2, XGB_MAX_DEPTH, False),
    "min_child_weight": ("float", 1.0, 30.0, True),
    "subsample":        ("float", 0.5, 1.0, False),
    "colsample_bytree": ("float", 0.5, 1.0, False),
    "gamma":            ("float", 1e-8, 5.0, True),
    "reg_alpha":        ("float", 1e-8, 10.0, True),
    "reg_lambda":       ("float", 1e-8, 10.0, True),
}
XGB_DISTS = {k: (IntDistribution if t == "int" else FloatDistribution)(lo, hi, log=lg) for k, (t, lo, hi, lg) in XGB_SPACE.items()}
XGB_FP = fingerprint("xgb-v2", SEED, OUTER_FOLDS, INNER_FOLDS, XGB_TRIALS, XGB_SPACE, MISSING_POLICY, OXIDE_TABLE_HASH,
                     SELF_TEST and SELFTEST_ROWS)

def make_xgb(params, seed=SEED):
    return xgb.XGBRegressor(objective="reg:squarederror", tree_method="hist", n_jobs=N_JOBS, random_state=seed, **params)

def suggest_xgb(trial):
    return {k: (trial.suggest_int if t == "int" else trial.suggest_float)(k, lo, hi, log=lg)
            for k, (t, lo, hi, lg) in XGB_SPACE.items()}

def tune_xgb(X, y, groups, n_trials, seed, trial_log, label=""):
    # Resumable Optuna search. Returns (best_params, best_inner_RMSE, n_trials_run_now).
    done = load_json(trial_log) or []
    finished = lambda: sum(r["state"] in ("COMPLETE", "PRUNED") for r in done)   # failed trials do not use up the budget
    remaining = n_trials - finished()
    if remaining > 0:
        inner = grouped_kfold(groups, INNER_FOLDS, seed)
        study = optuna.create_study(direction="minimize",
                                    sampler=optuna.samplers.TPESampler(seed=seed + len(done)),
                                    pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=1))
        for r in done:                                           # replay finished trials
            iv = {int(k): v for k, v in r["intermediate"].items()}
            dists = {k: XGB_DISTS[k] for k in r["params"]}
            if r["state"] == "COMPLETE":
                study.add_trial(create_trial(params=r["params"], distributions=dists, value=r["value"], intermediate_values=iv))
            elif r["state"] == "PRUNED":
                study.add_trial(create_trial(params=r["params"], distributions=dists, intermediate_values=iv, state=TrialState.PRUNED))
        if done:
            log(f"    {label}: resuming tuning, {finished()} trials loaded, {remaining} to go")

        def objective(trial):
            params, rmses = suggest_xgb(trial), []
            for step, (tr, va) in enumerate(inner):
                ensure_memory(label)
                try:
                    m = make_xgb(params, seed).fit(X[tr], y[tr])
                except XGB_ERRORS as e:
                    raise OutOfMemoryStop(str(e)[:120]) from None
                rmses.append(float(np.sqrt(mean_squared_error(y[va], m.predict(X[va])))))
                del m
                trial.report(float(np.mean(rmses)), step)
                if trial.should_prune():
                    gc.collect(); raise optuna.TrialPruned()
            gc.collect()
            return float(np.mean(rmses))

        def save_trial(study, trial):                            # called after every trial
            done.append({"params": trial.params, "value": trial.value, "state": trial.state.name,
                         "intermediate": {str(k): v for k, v in trial.intermediate_values.items()}})
            save_json(trial_log, done)
            if finished() % 10 == 0:
                free, proc = ram_status()
                ram = "" if free is None else f" | free RAM {free:.1f} GB, notebook uses {proc:.1f} GB"
                log(f"    {label}: {finished()}/{n_trials} trials{ram}")

        try:
            study.optimize(objective, n_trials=remaining, callbacks=[save_trial], show_progress_bar=False)
        except (OutOfMemoryStop, MemoryError) as e:
            stop_for_memory(label, e)
    complete = [r for r in done if r["state"] == "COMPLETE"]
    if not complete:
        raise RuntimeError(f"{label}: no Optuna trial completed.")
    best = min(complete, key=lambda r: r["value"])
    return best["params"], best["value"], max(remaining, 0)

def nested_cv_xgb(name, X, y, groups, outer, label):
    oof, rows, best = np.full(len(y), np.nan), [], []
    for k, (tr, te) in enumerate(outer):
        res_path = CKPT_DIR / name / f"{label}_outer{k}_{XGB_FP}.json"
        res, t0 = load_json(res_path), time.time()
        if res is not None and res.get("test_index_hash") != index_hash(te):
            raise RuntimeError(f"{res_path}: fold assignment changed; delete {CKPT_DIR} or use a new RESULTS_DIR")
        if res is None:
            params, inner_rmse, n_new = tune_xgb(X[tr], y[tr], groups[tr], XGB_TRIALS, SEED + k,
                                                 CKPT_DIR / name / f"{label}_outer{k}_trials_{XGB_FP}.json", f"{name}/{label}/fold{k}")
            try:
                ensure_memory(label); m = make_xgb(params).fit(X[tr], y[tr])
            except (OutOfMemoryStop, MemoryError) + XGB_ERRORS as e:
                stop_for_memory(f"{name}/{label}/fold{k} refit", e)
            res = {"pred": m.predict(X[te]).astype(float).tolist(), "params": params, "inner_cv_RMSE": inner_rmse,
                   "test_index_hash": index_hash(te), "minutes": (time.time() - t0) / 60}
            save_json(res_path, res); del m; gc.collect()
            note = f"({res['minutes']:.1f} min, saved)"
        else:
            note = "(loaded from disk)"
        oof[te] = np.asarray(res["pred"])
        rows.append({"model": label, "fold": k, **metrics(y[te], oof[te])})
        best.append({"fold": k, "inner_cv_RMSE": res["inner_cv_RMSE"], **res["params"]})
        log(f"  {name} {label:16s} fold {k}: test R²={rows[-1]['R2']:.4f} RMSE={rows[-1]['RMSE']:.4g} {note}")
    return oof, rows, pd.DataFrame(best)

In [ ]:
assert "CONFIG_LOADED" in globals(), "Run the notebook from the top: Restart Kernel -> Run All"
cv_rows, oof_preds, xgb_params = {}, {}, {}
for name, d in data.items():
    log(f"=== {name}: baselines" + (" + XGBoost ===" if RUN_XGB else " (XGBoost switched off) ==="))
    X, y, g = d["X"].to_numpy(), d["y"], d["groups"]
    rows, oof = [], {"mixing_rule": d["X"]["mean_p"].to_numpy(), "linear_5desc": np.full(len(y), np.nan)}
    for k, (tr, te) in enumerate(d["outer"]):
        oof["linear_5desc"][te] = LinearRegression().fit(X[tr], y[tr]).predict(X[te])
        rows.append({"model": "mixing_rule (mean_p)", "fold": k, **metrics(y[te], oof["mixing_rule"][te])})
        rows.append({"model": "linear_5desc", "fold": k, **metrics(y[te], oof["linear_5desc"][te])})
    if RUN_XGB:
        oof["xgb_5desc"], r, xgb_params[name] = nested_cv_xgb(name, X, y, g, d["outer"], "xgb_5desc"); rows += r
        if RUN_RAW_OXIDE_BASELINE:
            oof["xgb_raw_oxides"], r, _ = nested_cv_xgb(name, d["comp"].to_numpy(), y, g, d["outer"], "xgb_raw_oxides"); rows += r
        xgb_params[name].to_csv(RESULTS_DIR / f"xgb_best_params_per_fold_{name}.csv", index=False)
    cv_rows[name], oof_preds[name] = rows, oof
    display(summarize(rows))

## 7. Symbolic regression (PySR): nested grouped CV, resume per PySR run

For each outer fold:

1. The outer training part is split by groups into a fit set and a validation set.
2. PySR runs once per `SR_GRID` setting on the fit set. Every finished run (its Pareto front, validation scores and test-fold predictions) is saved immediately.
3. The simplest equation within `SR_TOLERANCE` of the best validation RMSE is chosen and scored on the outer test fold.

In [ ]:
assert "CONFIG_LOADED" in globals(), "Run the notebook from the top: Restart Kernel -> Run All"
if RUN_SR:
    from pysr import PySRRegressor
    import inspect
    _PYSR_ARGS = set(inspect.signature(PySRRegressor.__init__).parameters)

SR_FP = fingerprint("sr-v2", SEED, OUTER_FOLDS, SR_NITERATIONS, SR_TIMEOUT_MIN, SR_POPULATIONS, SR_POPULATION_SIZE, SR_GRID,
                    SR_BINARY_OPS, SR_UNARY_OPS, SR_VAL_FRACTION, MISSING_POLICY, OXIDE_TABLE_HASH, SELF_TEST and SELFTEST_ROWS)

def make_pysr(n_train, run_id, maxsize, parsimony, seed):
    kw = dict(niterations=SR_NITERATIONS, timeout_in_seconds=60 * SR_TIMEOUT_MIN,
              populations=SR_POPULATIONS, population_size=SR_POPULATION_SIZE,
              heap_size_hint_in_bytes=int(SR_JULIA_HEAP_GB * 1024**3),
              binary_operators=SR_BINARY_OPS, unary_operators=SR_UNARY_OPS,
              constraints={"^": (-1, 1)},
              nested_constraints={"exp": {"exp": 0, "log": 0}, "log": {"exp": 0, "log": 0}, "^": {"^": 0, "exp": 0}},
              turbo=True, bumper=True, maxsize=maxsize, parsimony=parsimony,
              elementwise_loss="L2DistLoss()", model_selection="accuracy",
              batching=n_train > 5000, batch_size=SR_BATCH_SIZE, random_state=seed,
              progress=False, verbosity=0, output_directory=str(RESULTS_DIR / "pysr_runs"), run_id=run_id)
    if SR_DETERMINISTIC:
        kw.update(deterministic=True, parallelism="serial")
    skipped = [k for k in kw if k not in _PYSR_ARGS]
    if skipped:
        log("  note: this PySR version does not support", skipped, "- skipped")
    return PySRRegressor(**{k: v for k, v in kw.items() if k in _PYSR_ARGS})

def equation_function(sympy_str):
    # numpy function of the five descriptors, built from the exported equation text
    expr = sp.sympify(sympy_str, locals={f: sp.Symbol(f) for f in FEATURES})
    f = sp.lambdify([sp.Symbol(c) for c in FEATURES], expr, "numpy")
    return lambda Xdf: np.broadcast_to(np.asarray(f(*[Xdf[c].to_numpy(float) for c in FEATURES]), float), (len(Xdf),))

def sr_run(Xdf, y, fit_idx, val_idx, cfg_i, seed, tag, ckpt, X_apply=None):
    # One PySR run (one SR_GRID setting); result saved/loaded as a unit.
    res = load_json(ckpt)
    if res is not None:
        return res, True
    cfg = SR_GRID[cfg_i]; t0 = time.time()
    m = make_pysr(len(fit_idx), f"{tag}_cfg{cfg_i}_{int(time.time())}", cfg["maxsize"], cfg["parsimony"], seed)
    m.fit(Xdf.iloc[fit_idx], y[fit_idx])
    eqs = m.equations_.reset_index(drop=True)
    front = []
    for i in range(len(eqs)):
        try:
            pv = np.asarray(m.predict(Xdf.iloc[val_idx], index=i), float)
            if not np.all(np.isfinite(pv)):
                continue
            row = {"cfg": cfg_i, "index": i, "complexity": int(eqs.loc[i, "complexity"]),
                   "equation": str(eqs.loc[i, "equation"]), "sympy": str(m.sympy(index=i)),
                   "val_RMSE": float(np.sqrt(mean_squared_error(y[val_idx], pv)))}
            if X_apply is not None:
                pa = np.asarray(m.predict(X_apply, index=i), float)
                row["apply_pred"] = pa.tolist() if np.all(np.isfinite(pa)) else None
            front.append(row)
        except Exception as e:
            log(f"    skipped equation {i}: {type(e).__name__}")
    res = {"front": front, "minutes": (time.time() - t0) / 60}
    save_json(ckpt, res)
    del m; gc.collect()
    return res, False

def sr_fit_select(name, Xdf, y, groups, seed, tag, X_apply=None):
    fit_idx, val_idx = grouped_split(groups, SR_VAL_FRACTION, seed)
    front = []
    for c in range(len(SR_GRID)):
        res, loaded = sr_run(Xdf, y, fit_idx, val_idx, c, seed, tag, CKPT_DIR / name / f"{tag}_cfg{c}_{SR_FP}.json", X_apply)
        log(f"    {tag} setting {c} {SR_GRID[c]}: " + ("loaded from disk" if loaded else f"{res['minutes']:.1f} min, saved"))
        front += res["front"]
    allf = pd.DataFrame(front)
    if X_apply is not None:
        allf = allf[allf["apply_pred"].notna()]
    best = allf["val_RMSE"].min()
    chosen = allf[allf["val_RMSE"] <= (1 + SR_TOLERANCE) * best].sort_values(["complexity", "val_RMSE"]).iloc[0]
    return chosen.to_dict(), allf.drop(columns=["apply_pred"], errors="ignore")

sr_selected, EQUATION_EXPORT_DIFF = {}, []
for name, d in (data.items() if RUN_SR else []):
    log(f"=== {name}: symbolic regression ===")
    Xdf, y, g = d["X"], d["y"], d["groups"]
    cv_rows[name] = [r for r in cv_rows[name] if r["model"] != "symbolic_regression"]
    oof_sr, picked = np.full(len(y), np.nan), []
    for k, (tr, te) in enumerate(d["outer"]):
        chosen, _ = sr_fit_select(name, Xdf.iloc[tr].reset_index(drop=True), y[tr], g[tr], SEED + k,
                                  f"sr_outer{k}", X_apply=Xdf.iloc[te])
        oof_sr[te] = chosen["apply_pred"]
        # the exported equation text must reproduce PySR's own predictions
        re_eval = equation_function(chosen["sympy"])(Xdf.iloc[te])
        rel = float(np.max(np.abs(re_eval - oof_sr[te]) / (np.abs(oof_sr[te]) + 1e-9 * np.abs(y).max())))
        EQUATION_EXPORT_DIFF.append(rel)
        if not rel < 1e-3:
            log(f"  WARNING: exported equation differs from PySR prediction (max rel. diff {rel:.2g})")
        cv_rows[name].append({"model": "symbolic_regression", "fold": k, **metrics(y[te], oof_sr[te])})
        picked.append({"fold": k, "setting": SR_GRID[int(chosen["cfg"])], "complexity": int(chosen["complexity"]),
                       "val_RMSE": chosen["val_RMSE"], "equation": chosen["sympy"]})
        log(f"  {name} SR fold {k}: test R²={cv_rows[name][-1]['R2']:.4f} complexity={picked[-1]['complexity']}  {chosen['sympy']}")
    oof_preds[name]["symbolic_regression"] = oof_sr
    sr_selected[name] = pd.DataFrame(picked)
    sr_selected[name].to_csv(RESULTS_DIR / f"sr_selected_per_fold_{name}.csv", index=False)

## 8. Cross-validated results (the numbers to report)

In [ ]:
assert "CONFIG_LOADED" in globals(), "Run the notebook from the top: Restart Kernel -> Run All"
summary = []
for name, rows in cv_rows.items():
    for model_name, r in summarize(rows).iterrows():
        summary.append({"property": name, "model": model_name,
                        **{f"{m} mean": r[(m, "mean")] for m in ["R2", "RMSE", "MAE", "MedAE"]},
                        **{f"{m} std": r[(m, "std")] for m in ["R2", "RMSE", "MAE", "MedAE"]}})
    pd.DataFrame(rows).to_csv(RESULTS_DIR / f"cv_fold_metrics_{name}.csv", index=False)
    pd.DataFrame({"y_true": data[name]["y"], **oof_preds[name]}).to_csv(RESULTS_DIR / f"oof_predictions_{name}.csv", index=False)
summary = pd.DataFrame(summary)
summary.to_csv(RESULTS_DIR / "cv_summary.csv", index=False)
fmt = summary.copy()
for m in ["R2", "RMSE", "MAE", "MedAE"]:
    fmt[m] = fmt[f"{m} mean"].map("{:.4g}".format) + " ± " + fmt[f"{m} std"].map("{:.2g}".format)
display(fmt[["property", "model", "R2", "RMSE", "MAE", "MedAE"]])

for name in data:
    y = data[name]["y"]
    models = [m for m in ["xgb_5desc", "symbolic_regression", "xgb_raw_oxides"] if m in oof_preds[name]]
    if not models:
        continue
    fig, axes = plt.subplots(1, len(models), figsize=(4 * len(models), 4)); axes = np.atleast_1d(axes)
    lo, hi = np.percentile(y, [0.1, 99.9])
    for ax, m in zip(axes, models):
        r = metrics(y, oof_preds[name][m])
        ax.scatter(y, oof_preds[name][m], s=3, alpha=0.35); ax.plot([lo, hi], [lo, hi], "k--", lw=1)
        ax.set_title(f"{m}\nOOF R²={r['R2']:.3f}, RMSE={r['RMSE']:.3g}", fontsize=9)
        ax.set_xlabel(f"measured ({TARGETS[name]['unit']})"); ax.set_ylabel("predicted (out-of-fold)")
        ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
    fig.suptitle(name); fig.tight_layout(); fig.savefig(RESULTS_DIR / f"parity_{name}.png"); plt.show(); plt.close(fig)

## 9. Final models on all data

These are the models to use and interpret. Their performance is the cross-validated estimate above.

In [ ]:
assert "CONFIG_LOADED" in globals(), "Run the notebook from the top: Restart Kernel -> Run All"

class SavedBooster:
    def __init__(self, path):
        self.booster = xgb.Booster(); self.booster.load_model(str(path))
    def predict(self, X):
        return np.asarray(self.booster.inplace_predict(np.asarray(X, dtype=float)), float)

final = {}
for name, d in (data.items() if RUN_FINAL else []):
    log(f"=== {name}: final models ===")
    Xdf, X, y, g = d["X"], d["X"].to_numpy(), d["y"], d["groups"]
    entry = {}
    if RUN_XGB:
        model_path = CKPT_DIR / name / f"xgb_final_{XGB_FP}.json"
        meta_path = CKPT_DIR / name / f"xgb_final_meta_{XGB_FP}.json"
        if load_json(meta_path) is None or not model_path.exists():
            params, inner_rmse, _ = tune_xgb(X, y, g, XGB_TRIALS, SEED, CKPT_DIR / name / f"xgb_final_trials_{XGB_FP}.json", f"{name}/final")
            m = make_xgb(params).fit(X, y)
            m.get_booster().save_model(str(model_path))          # native format (works with xgboost 3 + scikit-learn >= 1.6)
            assert np.allclose(SavedBooster(model_path).predict(X[:200]), m.predict(X[:200]), rtol=1e-5, atol=1e-6)
            save_json(meta_path, {"params": params, "inner_cv_RMSE": inner_rmse}); del m; gc.collect()
        meta = load_json(meta_path)
        entry = {"xgb": SavedBooster(model_path), "xgb_params": meta["params"]}
        shutil.copyfile(model_path, RESULTS_DIR / f"xgb_final_{name}.json")
        save_json(RESULTS_DIR / f"xgb_final_params_{name}.json", meta["params"])
        log(f"  XGBoost final model saved (inner-CV RMSE {meta['inner_cv_RMSE']:.4g})")
    if RUN_SR:
        chosen, front = sr_fit_select(name, Xdf, y, g, SEED, "sr_final")
        front.to_csv(RESULTS_DIR / f"sr_final_pareto_{name}.csv", index=False)
        entry.update(sr_chosen=chosen, sr_front=front)
        log(f"  final equation (complexity {int(chosen['complexity'])}): {chosen['sympy']}")
    final[name] = entry

In [ ]:
assert "CONFIG_LOADED" in globals(), "Run the notebook from the top: Restart Kernel -> Run All"
eq_rows = []
for name, f in final.items():
    if "sr_chosen" not in f:
        continue
    ch = f["sr_chosen"]
    expr = sp.sympify(ch["sympy"], locals={c: sp.Symbol(c) for c in FEATURES})
    pred_all = equation_function(ch["sympy"])(data[name]["X"])
    eq_rows.append({"property": name, "unit": TARGETS[name]["unit"], "complexity": int(ch["complexity"]),
                    "equation": str(expr), "latex": sp.latex(expr), "R2_all_data(in-sample)": r2_score(data[name]["y"], pred_all)})
    fr = f["sr_front"]
    fig, ax = plt.subplots(figsize=(4.8, 3.3))
    for c, sub in fr.groupby("cfg"):
        sub = sub.sort_values("complexity")
        ax.plot(sub["complexity"], sub["val_RMSE"], "o-", ms=3, label=str(SR_GRID[int(c)]))
    ax.axvline(ch["complexity"], color="k", ls=":", lw=1)
    ax.set_xlabel("equation complexity"); ax.set_ylabel(f"validation RMSE ({TARGETS[name]['unit']})")
    ax.set_title(f"{name}: Pareto fronts"); ax.legend(fontsize=6)
    fig.tight_layout(); fig.savefig(RESULTS_DIR / f"sr_pareto_{name}.png"); plt.show(); plt.close(fig)
equations = pd.DataFrame(eq_rows)
equations.to_csv(RESULTS_DIR / "final_equations.csv", index=False)
for _, r in equations.iterrows():
    print(f"\n{r['property']} [{r['unit']}]  (complexity {r['complexity']})\n  {r['equation']}")
equations

In [ ]:
assert "CONFIG_LOADED" in globals(), "Run the notebook from the top: Restart Kernel -> Run All"

def permutation_rmse_increase(model, X, y, n_repeats=10, seed=SEED):
    rng = np.random.default_rng(seed)
    base = np.sqrt(mean_squared_error(y, model.predict(X)))
    out = []
    for j in range(X.shape[1]):
        inc = []
        for _ in range(n_repeats):
            Xp = X.copy(); Xp[:, j] = rng.permutation(Xp[:, j])
            inc.append(np.sqrt(mean_squared_error(y, model.predict(Xp))) - base)
        out.append(np.mean(inc))
    return np.array(out)

for name, f in final.items():
    if "xgb" not in f:
        continue
    imp = pd.Series(permutation_rmse_increase(f["xgb"], data[name]["X"].to_numpy(), data[name]["y"]), index=FEATURES).sort_values()
    imp.to_csv(RESULTS_DIR / f"permutation_importance_{name}.csv")
    fig, ax = plt.subplots(figsize=(4, 2.5)); imp.plot.barh(ax=ax, title=f"{name}: permutation importance (ΔRMSE)")
    fig.tight_layout(); fig.savefig(RESULTS_DIR / f"permutation_importance_{name}.png"); plt.show(); plt.close(fig)

## 10. Run record

In [ ]:
assert "CONFIG_LOADED" in globals(), "Run the notebook from the top: Restart Kernel -> Run All"
save_json(RESULTS_DIR / "run_record.json", {
    "environment": ENV, "descriptor_build": build_report,
    "config": dict(OUTER_FOLDS=OUTER_FOLDS, INNER_FOLDS=INNER_FOLDS, XGB_TRIALS=XGB_TRIALS, XGB_SPACE=XGB_SPACE,
                   SR_NITERATIONS=SR_NITERATIONS, SR_TIMEOUT_MIN=SR_TIMEOUT_MIN, SR_GRID=SR_GRID, SR_TOLERANCE=SR_TOLERANCE,
                   SR_VAL_FRACTION=SR_VAL_FRACTION, SR_BINARY_OPS=SR_BINARY_OPS, SR_UNARY_OPS=SR_UNARY_OPS,
                   MISSING_POLICY=MISSING_POLICY, PROPERTY_OVERRIDES=PROPERTY_OVERRIDES, SEED=SEED, SELF_TEST=SELF_TEST),
    "dataset_md5": {n: md5sum(p) for n, p in DATA_FILES.items()}, "oxide_table_md5": md5sum(OXIDE_TABLE)})
log("run record saved. All outputs are in", RESULTS_DIR.resolve())

## 11. Self-test (runs only when `SELF_TEST = True`)

This repeats parts of the work and checks that:
- finished trials, folds and PySR runs are loaded from disk, not recomputed;
- a tuning run interrupted part-way resumes with only the missing trials;
- loaded results are identical to the originals.

In [ ]:
assert "CONFIG_LOADED" in globals(), "Run the notebook from the top: Restart Kernel -> Run All"
if SELF_TEST:
    checks = []
    def check(name, ok, detail=""):
        checks.append({"check": name, "result": "PASS" if ok else "FAIL", "detail": detail})

    name0 = list(data)[0]; d0 = data[name0]
    X0, y0, g0 = d0["X"].to_numpy(), d0["y"], d0["groups"]

    check("descriptors: hand-computed examples + AM >= GM >= HM", True, "asserted while building")
    check("folds: no group on both sides; every row tested once", True, "asserted while building")

    if RUN_XGB:
        # 1) fold-level resume: everything must load, predictions identical
        t0 = time.time()
        oof2, _, _ = nested_cv_xgb(name0, X0, y0, g0, d0["outer"], "xgb_5desc")
        check("XGBoost folds reload from disk", np.allclose(oof2, oof_preds[name0]["xgb_5desc"]) and time.time() - t0 < 30,
               f"{time.time() - t0:.1f} s, identical predictions")

        # 2) trial-level resume: run 3 trials, 'crash', then ask for 6 -> only 3 new
        tl = RESULTS_DIR / "selftest_trials.json"
        _, _, n1 = tune_xgb(X0, y0, g0, 3, 999, tl, "selftest")
        rec = load_json(tl)
        _, best_v, n2 = tune_xgb(X0, y0, g0, 6, 999, tl, "selftest")
        rec2 = load_json(tl)
        check("Optuna resumes at trial level", n1 == 3 and n2 == 3 and len(rec2) == 6 and rec2[:3] == rec,
               f"first run {n1} trials, resumed run {n2} new trials, log has {len(rec2)}")
        _, _, n3 = tune_xgb(X0, y0, g0, 6, 999, tl, "selftest")
        check("finished tuning is not repeated", n3 == 0, f"{n3} trials run on re-call")

        # 3) interrupted tuning inside an outer fold: delete fold result, keep half the trial log
        lbl = "xgb_5desc"; k = 0
        res_path = CKPT_DIR / name0 / f"{lbl}_outer{k}_{XGB_FP}.json"
        trial_path = CKPT_DIR / name0 / f"{lbl}_outer{k}_trials_{XGB_FP}.json"
        full_log = load_json(trial_path); res_path.unlink()
        save_json(trial_path, full_log[: len(full_log) // 2])
        oof3, _, _ = nested_cv_xgb(name0, X0, y0, g0, d0["outer"], lbl)
        check("interrupted fold resumes with remaining trials",
               sum(r["state"] != "FAIL" for r in load_json(trial_path)) == XGB_TRIALS and res_path.exists(),
               f"{len(full_log) // 2} trials kept, now {len(load_json(trial_path))}")

    # 4) PySR runs reload, exported equations reproduce PySR predictions
    if RUN_SR:
        tr, te = d0["outer"][0]
        t0 = time.time()
        ch, _ = sr_fit_select(name0, d0["X"].iloc[tr].reset_index(drop=True), y0[tr], g0[tr], SEED, "sr_outer0", X_apply=d0["X"].iloc[te])
        check("PySR runs reload from disk", np.allclose(ch["apply_pred"], oof_preds[name0]["symbolic_regression"][te]) and time.time() - t0 < 30,
              f"{time.time() - t0:.1f} s")
        check("exported equations reproduce PySR predictions", len(EQUATION_EXPORT_DIFF) > 0 and max(EQUATION_EXPORT_DIFF) < 1e-3,
              f"max relative difference {max(EQUATION_EXPORT_DIFF):.1e}")
    # 5) final XGBoost model re-loads with identical predictions
    if RUN_FINAL and RUN_XGB:
        check("final XGBoost model saved and re-loaded", (CKPT_DIR / name0 / f"xgb_final_{XGB_FP}.json").exists(),
              "predictions compared on save")
    if RUN_FINAL:
        check("final equations parsed by SymPy", RUN_SR and len(equations) == len(final), f"{len(equations)} equations")

    for fname in ["cv_summary.csv", "run_record.json", "progress.log"]:
        check(f"output file {fname}", (RESULTS_DIR / fname).exists())

    report = pd.DataFrame(checks)
    display(report)
    if (report["result"] == "FAIL").any():
        raise RuntimeError("SELF-TEST FAILED -- see the table above.")
    log("SELF-TEST PASSED. Set SELF_TEST = True  in the configuration cell and Run All for the full run.")
else:
    print("Full run finished. (Self-test is skipped when SELF_TEST = False.)")

## 12. Methods text (template, edit to match your final run)

> Each glass composition was converted to mole fractions $x_i$ over 17 oxides. For each target property, five descriptors were computed from the corresponding pure-oxide property $p_i$ (Young's modulus, density or melting temperature; values and sources in Table S1):
> - the composition-weighted arithmetic, geometric and harmonic means of $p_i$;
> - the composition-weighted variance of $p_i$;
> - the ideal mixing entropy $-\sum x_i \ln x_i$.
>
> Compositions containing an oxide without a literature value were excluded (Table S2).
>
> Generalisation error was estimated by nested *k*-fold cross-validation (5 outer, 5 inner folds). Rows with identical descriptor vectors were kept in the same fold. XGBoost hyperparameters were optimised in the inner loop by Bayesian optimisation (Optuna TPE with median pruning, 100 trials, minimising RMSE).
>
> Symbolic regression used PySR with the operators {+, −, ×, ÷, ^, square, cube, √, log, exp}, up to 1000 iterations and a 20 min limit per run. Its search settings and final equation complexity were chosen on a grouped inner validation split: the simplest Pareto-front equation within 1 % of the lowest validation RMSE.
>
> All reported metrics (R², RMSE, MAE, MedAE; mean ± s.d. over outer folds) were computed on outer test folds not used for any fitting or selection. Baselines evaluated on identical folds were the arithmetic-mean mixing rule, ordinary least squares on the five descriptors, and XGBoost on the raw oxide fractions.